In [1]:
# A1 · Part 2 — Build a minimal retrieval system.  Starter notebook.
# PE6201 · Emerging AI Technologies. Run top to bottom, then replace the placeholder domain.

# A1 · Part 2 — Build a minimal retrieval system  *(starter)*

**What Part 2 asks of you.** Assemble a small collection of documents from **your** domain —
10 to 30 is enough. Build a system that retrieves relevant passages and uses them to answer
questions. Show it answering **five questions it could not possibly answer without your
documents**. Then **break it**: find a question where retrieval fails or returns the wrong
passage, and explain *why* it failed.

> **The breaking is not optional and it is not a penalty.** Finding the edge of your own system
> is the single most valuable thing you will do in this assignment. A notebook where everything
> works is a notebook that was not pushed hard enough.

**What this notebook is.** A working retrieval-augmented generation (RAG) pipeline on a
**placeholder domain** (houseplant care notes). Everything you must change is marked
**`### REPLACE ME`**.

**Cost.** About 10 short generation calls. Fractions of a cent. Embeddings run locally and free.

**Deliverable from this notebook:** this notebook (run, output visible), five worked examples,
one documented failure, and roughly 400 words.

> **If a cell does not work.** This notebook was written before you ran it, against library
> versions, model names and provider behaviour that change without notice — so treat it as a
> working draft, not gospel. **Read the error, then try to fix it yourself:** debugging is part of
> the work, not a detour from it. Then **report it** — in class or by email — with the cell, the
> full error text, and what you tried, so the shared copy gets fixed and nobody behind you loses
> the same hour. Reporting a problem costs you nothing. The one thing not to do is quietly work
> around a broken cell: if your fix changes what the notebook measures, say so in your writeup.

### Setup

In [2]:
%pip -q install openai scikit-learn sentence-transformers

### Connect to OpenRouter
The **generation** step (writing the answer) goes to OpenRouter with your key. The
**embedding** step (turning text into vectors) runs **locally and free** — OpenRouter serves chat
models, not embedding models, which is exactly the "which layer do you rent?" question from
Class 2.

In [3]:
import os, getpass, time
import numpy as np
from openai import OpenAI

def _get_openrouter_key():
    k = os.environ.get('MY_PRIVATE_OPENROUTER_KEY') or os.environ.get('OPENROUTER_API_KEY')
    if k: return k
    try:
        from google.colab import userdata
        k = userdata.get('OPENROUTER_API_KEY')
        if k: return k
    except Exception:
        pass
    return getpass.getpass('Paste your OpenRouter API key (hidden): ')

API_KEY = _get_openrouter_key()
client  = OpenAI(base_url='https://openrouter.ai/api/v1', api_key=API_KEY)
GEN_MODEL = 'openai/gpt-4o-mini'      # TRY THIS (knob): any model your key can reach
TOKENS = {'in': 0, 'out': 0}

def generate(prompt, system=None, max_new_tokens=300, temperature=0.0):
    msgs = ([{'role': 'system', 'content': system}] if system else []) + \
           [{'role': 'user', 'content': prompt}]
    r = client.chat.completions.create(model=GEN_MODEL, messages=msgs,
                                       temperature=temperature, max_tokens=max_new_tokens)
    if getattr(r, 'usage', None):
        TOKENS['in']  += r.usage.prompt_tokens
        TOKENS['out'] += r.usage.completion_tokens
    return r.choices[0].message.content.strip()

print('connected ·', generate('Reply with exactly the word: ready', max_new_tokens=6))

Paste your OpenRouter API key (hidden): ··········
connected · ready


### Check your remaining credit  —  free, this call is not billed
Your allocation is **fixed and will not be topped up**, so watch it rather than discovering it at
the end. `usage` is what you have spent; `limit` is your cap; the difference is what is left.
Run this at the start of a working session and again at the end.

In [4]:
import urllib.request, json as _json

def credit_check():
    """Ask OpenRouter what is left on your key. Costs nothing."""
    try:
        req = urllib.request.Request('https://openrouter.ai/api/v1/key',
                                     headers={'Authorization': f'Bearer {API_KEY}'})
        d = _json.load(urllib.request.urlopen(req, timeout=20))['data']
        used, cap = d.get('usage'), d.get('limit')
        if cap is None:
            print(f'spent so far: ${used:.4f}   (no cap set on this key)')
        else:
            print(f'spent ${used:.4f} of ${cap:.2f}  ->  ${cap - used:.4f} remaining')
        rl = d.get('rate_limit') or {}
        if rl:
            print(f"rate limit: {rl.get('requests')} requests per {rl.get('interval')}")
    except Exception as e:
        print('could not check credit (not a problem, carry on):', str(e)[:120])

credit_check()

spent $0.0066 of $10.00  ->  $9.9934 remaining
rate limit: -1 requests per 10s


---
## 0 · Five words you need — *start here if you don't code*

| Word | In plain words |
|---|---|
| **Embedding** | A piece of text turned into a list of numbers, arranged so that texts with similar *meaning* end up near each other. |
| **Chunk** | A small slice of a document. We search chunks, not whole documents, so what comes back is the relevant *paragraph* rather than the whole manual. |
| **Retrieval** | Finding the chunks nearest the question. Pure arithmetic — no model involved, no understanding. |
| **Top-k** | How many chunks we fetch. Too few and the answer is missing; too many and the real answer drowns in noise. |
| **Grounding** | Making the model answer **only** from the retrieved text, and say so when the text does not cover it. |

**RAG in one line:** *embed your documents → retrieve the ones nearest the question → let a model
answer from them.* **The model is rented; the retrieval is yours.** That is why retrieval is where
almost every failure lives — and why the failure you document is the interesting part.

---
## 1 · Your documents  ### REPLACE ME

**10 to 30 documents.** Each a short paragraph — a few sentences is fine.

**The one rule that matters:** they must contain things a model *could not already know*. If your
documents are summaries of Wikipedia, the model answers correctly with retrieval switched off, and
you have proved nothing. Good sources: your own notes, internal procedures, club or team records,
a rulebook nobody has digitised, your family's version of anything, recent local information,
specifics with names and numbers in them.

**Test for this later, in Section 4.** We ask the model each question with the documents removed.
If it still gets them right, your document set is too generic — go back and make it specific.

In [5]:
"""PE6201 Part 2: 20 source-grounded English cat-care notes.

使用方法：将本文件完整复制到原 Notebook 的 DOCS 单元，替换旧内容。
运行后继续执行原分块、嵌入和检索单元。无需安装额外依赖。
DOCS 保持原来的 list[str] 格式；DOC_META 用原检索返回的零起始 doc_id 查来源。
例如：DOC_META[0] 对应 CAT01，而不是 CAT00。

整理日期：2026-10-02。20 条短资料来自 15 个网页，不是 20 个独立来源。
英文正文为根据网页重新表述的摘要，不是逐字摘录；中文说明仅用于理解。
category 是本项目整理资料时分配的主题标签，不是网站标签或咨询评测答案。
accessed_date 是访问日期，不能当作网页发布日期。未知更新日期保留 None。
公开资料不能保证 GPT 从未见过；仍须运行有资料/无资料的对照。
本数据仅提供正文和来源，不会自动给原 answer() 加引用，也不实现红旗规则。
"""

ACCESSED_DATE = "2026-10-02"

"""15个来源网页"""
SOURCE_PAGES = {
    "feeding": {
        "publisher": "Cats Protection",
        "source_title": "Feeding Your Cat",
        "url": "https://www.cats.org.uk/help-and-advice/diet/feeding",
        "updated_date": None,
    },
    "drinking": {
        "publisher": "Cats Protection",
        "source_title": "Cats and Drinking",
        "url": "https://www.cats.org.uk/help-and-advice/diet/cats-and-drinking",
        "updated_date": None,
    },
    "toileting": {
        "publisher": "Cats Protection",
        "source_title": "Cats & Toileting",
        "url": "https://www.cats.org.uk/help-and-advice/cat-behaviour/toileting",
        "updated_date": None,
    },
    "grooming": {
        "publisher": "Cats Protection",
        "source_title": "Grooming Your Cat",
        "url": "https://www.cats.org.uk/help-and-advice/cat-behaviour/grooming",
        "updated_date": None,
    },
    "bringing_home": {
        "publisher": "Cats Protection",
        "source_title": "Bringing a Cat Home",
        "url": "https://www.cats.org.uk/help-and-advice/home-and-environment/bringing-a-cat-home",
        "updated_date": None,
    },
    "scratching": {
        "publisher": "Cats Protection",
        "source_title": "Cats and Scratching",
        "url": "https://www.cats.org.uk/help-and-advice/cat-behaviour/scratching",
        "updated_date": None,
    },
    "play": {
        "publisher": "Cats Protection",
        "source_title": "Cats and Play",
        "url": "https://www.cats.org.uk/help-and-advice/cat-behaviour/cats-and-play",
        "updated_date": None,
    },
    "introductions": {
        "publisher": "Cats Protection",
        "source_title": "Introducing Cats & Kittens",
        "url": "https://www.cats.org.uk/help-and-advice/cat-behaviour/introducing-cats",
        "updated_date": None,
    },
    "fighting": {
        "publisher": "Cats Protection",
        "source_title": "Cats and Fighting",
        "url": "https://www.cats.org.uk/help-and-advice/cat-behaviour/cats-and-fighting",
        "updated_date": None,
    },
    "carrier": {
        "publisher": "Cats Protection",
        "source_title": "Choosing and Using a Cat Carrier",
        "url": "https://www.cats.org.uk/help-and-advice/getting-a-cat/how-to-choose-and-use-a-cat-carrier",
        "updated_date": None,
    },
    "home_safety": {
        "publisher": "Cats Protection",
        "source_title": "Preparing Your Home For a Cat",
        "url": "https://www.cats.org.uk/adopt-a-cat/preparing-for-a-cat",
        "updated_date": None,
    },
    "cornell_feeding": {
        "publisher": "Cornell University College of Veterinary Medicine",
        "source_title": "Feeding Your Cat",
        "url": "https://www.vet.cornell.edu/departments-centers-and-institutes/cornell-feline-health-center/health-information/feline-health-topics/feeding-your-cat",
        "updated_date": "2026-04-30",
    },
    "anorexia": {
        "publisher": "Cornell University College of Veterinary Medicine",
        "source_title": "Anorexia",
        "url": "https://www.vet.cornell.edu/departments-centers-and-institutes/cornell-feline-health-center/health-information/feline-health-topics/anorexia",
        "updated_date": None,
    },
    "dyspnea": {
        "publisher": "Cornell University College of Veterinary Medicine",
        "source_title": "Dyspnea (Difficulty Breathing)",
        "url": "https://www.vet.cornell.edu/departments-centers-and-institutes/cornell-feline-health-center/health-information/feline-health-topics/dyspnea-difficulty-breathing",
        "updated_date": None,
    },
    "urinary": {
        "publisher": "Cornell University College of Veterinary Medicine",
        "source_title": "Feline Lower Urinary Tract Disease",
        "url": "https://www.vet.cornell.edu/departments-centers-and-institutes/cornell-feline-health-center/health-information/feline-health-topics/feline-lower-urinary-tract-disease",
        "updated_date": "2016-10",
    },
}

#一个“资料整理函数”：把一条资料的正文、标题、类别和来源，统一装进一张资料卡。
def _doc(stable_id, title, title_cn, category, topic, source_key, section, text, summary_cn):
    return {
        "stable_id": stable_id,
        "title": title,
        "title_cn": title_cn,
        "category": category,
        "topic": topic,
        "language": "en",
        "text": " ".join(text.split()),
        "summary_cn": summary_cn,
        **SOURCE_PAGES[source_key],
        "source_section": section,
        "accessed_date": ACCESSED_DATE,
        "content_type": "source-grounded paraphrase",
    }

#20个知识库
KNOWLEDGE_BASE = [
    _doc("CAT01", "Complete and complementary cat food", "完整主食与补充食品", "routine_care", "food_selection", "feeding", "What should I feed my cat?",
         """Cats Protection recommends food labelled complete because it supplies the nutrients a cat needs. Complementary products, including treats, do not provide a balanced diet by themselves. Food should suit the cat's life stage. Homemade meals are not recommended unless a veterinarian advises them for a medical reason.""",
         "选标明 complete 的食品；complementary 食品不能单独充当均衡主食。"),
    _doc("CAT02", "Changing a cat's food gradually", "逐步换粮", "routine_care", "diet_transition", "feeding", "Changing your cat's food",
         """Cats Protection advises offering a small amount of new food beside the usual food in separate bowls, rather than mixing them. If the cat accepts it, increase the new food and reduce the old food over a week or longer. If the cat refuses it or becomes unwell, retain the old food and seek veterinary advice.""",
         "该指南建议新旧粮分碗提供，并用一周或更久逐步过渡；不接受或不适时咨询兽医。"),
    _doc("CAT03", "Water bowls and drinking preferences", "饮水碗与位置", "routine_care", "water_setup", "drinking", "How much should my cat be drinking?",
         """Cats Protection suggests wide ceramic, metal or glass water bowls so whiskers are not pressed against the sides. Keep water separate from food and litter trays, and offer bowls in several locations. Some cats prefer moving water. A sudden change in drinking should be checked by a veterinarian.""",
         "宽口陶瓷、金属或玻璃碗，多处提供并远离食物和猫砂盆；饮水突然变化需检查。"),
    _doc("CAT04", "Water rather than milk", "水与牛奶", "routine_care", "milk_and_water", "drinking", "What do cats drink?; Do cats drink milk?",
         """Cats need continuous access to clean drinking water. Cats Protection advises against cow's milk and human milk alternatives: many cats cannot digest lactose well, and alternatives may contain unsuitable ingredients. Commercial cat milk is an occasional high-calorie treat, not a replacement for water or a balanced diet.""",
         "水应持续供应；牛奶及人用替代奶不推荐，猫奶也不能代替水或均衡饮食。"),
    _doc("CAT05", "Litter tray quantity and placement", "猫砂盆数量与位置", "routine_care", "litter_setup", "toileting", "Where to put your cat's litter tray; The best litter tray for your cat",
         """Cats Protection recommends one litter tray for each cat plus one additional tray. Put trays in separate quiet, accessible locations, away from food, water and sleeping areas. Avoid noisy appliances and busy passageways. A row of neighbouring trays is not equivalent to distributing them. Cats with mobility difficulties benefit from low-sided trays.""",
         "每只猫一个盆，再额外加一个；分散放在安静、易到达的位置，行动困难的猫用低边盆。"),
    _doc("CAT06", "Cleaning litter trays", "猫砂盆清洁", "routine_care", "litter_cleaning", "toileting", "How to clean your cat's litter tray",
         """Cats Protection advises removing urine and faeces twice daily, or more often if needed. Empty and wash the tray with hot soapy water or a cat-safe cleaner at least weekly. Covered trays may need thorough cleaning more often because they retain smells. Avoid newspaper or plastic liners inside the tray.""",
         "每天至少清理排泄物两次，每周至少彻底清洗一次；封闭盆可能需更频繁清洗。"),
    _doc("CAT07", "Introducing gentle grooming", "温和梳毛", "routine_care", "grooming", "grooming", "Do I need to groom my cat?; How to get your cat used to being groomed",
         """Long-haired cats and cats unable to groom themselves may need brushing assistance. Cats Protection recommends introducing brushes gradually, working a little at a time and using minimal restraint. A sudden increase or decrease in self-grooming warrants a veterinary appointment, rather than assuming it is only a grooming preference.""",
         "梳毛应逐步适应、少量进行，减少强制固定；自我梳理突然增减应预约兽医。"),
    _doc("CAT08", "A quiet room for a newly adopted cat", "新猫的安静空间", "routine_care", "new_cat_setup", "bringing_home", "Setting up your cat's new space",
         """Before releasing a new cat from its carrier, Cats Protection recommends preparing a quiet room. Include separate food and water areas, a private litter tray far from both, a hiding place, a secure elevated resting spot, a bed, a scratching post and toys. This provides space to settle before exploring the whole home.""",
         "先准备安静房间，配齐分开的食水、猫砂盆、躲藏处、高处、床、抓柱和玩具。"),
    _doc("CAT09", "Allowing a new cat time to settle", "给新猫适应时间", "behaviour", "settling_in", "bringing_home", "When your cat first arrives home; Signs your cat is struggling to settle",
         """Cats Protection advises patience when a cat enters a new home; do not pressure it to interact or explore before it is ready. Initial hiding or avoidance can occur during adjustment. Look for gradual improvement rather than expecting every cat to settle at the same speed. Prolonged stress without improvement should prompt veterinary advice.""",
         "不要强迫新猫互动，不同猫适应速度不同；观察逐步改善，持续压力无改善需咨询兽医。"),
    _doc("CAT10", "Choosing a scratching surface", "选择抓挠用品", "behaviour", "scratching", "scratching", "Prevent your cat from scratching your furniture",
         """For claw-maintenance scratching, Cats Protection recommends a stable post tall enough for a full stretch, initially beside the scratched furniture. A horizontal pad can suit cats that scratch carpets. Once accepted, move it gradually. Provide separate posts for different cats; scratching linked to anxiety also requires attention to the source of stress.""",
         "抓柱需稳定且足够高，先放在被抓家具附近；偏好横向抓挠的猫可用抓板。"),
    _doc("CAT11", "Short play sessions and toy safety", "短时玩耍与玩具安全", "behaviour", "play_routine", "play", "How to play with your cat; How often should I play with my cat?",
         """Cats Protection says games lasting one or two minutes can be useful. Rotate toys to sustain interest and let a cat catch a wand toy occasionally. Inspect toys for damage. Do not leave a cat unsupervised with toys it might shred, swallow or become tangled in.""",
         "一两分钟的短游戏也有价值，轮换玩具并让猫偶尔捕获；有吞咽或缠绕风险的玩具需看护。"),
    _doc("CAT12", "Redirecting rough play away from hands", "避免把手脚当玩具", "behaviour", "rough_play", "play", "How can I teach my cat to play gently?; Is my cat's play too rough?",
         """Cats Protection discourages using fingers or toes as play targets. Use toys that keep hunting activity away from the body, such as wand toys or thrown balls. Regular play is particularly important indoors. Newly appearing play aggression in an adult cat should be checked by a veterinarian before seeking behavioural help.""",
         "用逗猫棒或球代替手脚；成年猫突然出现攻击性玩耍时，先排查医疗原因。"),
    _doc("CAT13", "Introducing cats in stages", "分阶段介绍新旧猫", "behaviour", "cat_introductions", "introductions", "Introducing cats or kittens: the basic steps",
         """Cats Protection describes staged introductions: a separate sanctuary room, scent exchange using cloths, visual contact through a glass barrier, then mesh-barrier contact before short face-to-face meetings. Progress only when cats are comfortable with the preceding stage. Do not force approach or meetings, and provide ways to retreat or reach a high place.""",
         "先独立房间，再交换气味、隔玻璃与网栏接触，最后短时见面；不强迫并保留退路。"),
    _doc("CAT14", "Reducing competition in multi-cat homes", "减少多猫资源竞争", "behaviour", "multi_cat_resources", "fighting", "Cats living together; How to help your cats get along",
         """Cats Protection recommends separate food and water bowls, beds, litter trays and hiding or sleeping places for each cat, with spare resources. Avoid placing essentials in busy shared areas or too close to another cat. Cats can experience tension without fighting. Sudden aggression in a normally calm cat should be discussed with a veterinarian.""",
         "为各猫分开配置资源并留备用；没有打架也可能有压力，突然攻击应咨询兽医。"),
    _doc("CAT15", "Selecting a secure cat carrier", "选择安全航空箱", "routine_care", "carrier_selection", "carrier", "What is the best type of cat carrier to buy?; Why do you need a cat carrier?",
         """Cats Protection recommends a sturdy, secure carrier that is easy to clean and allows comfortable lying and repositioning. Ventilation openings should be present on at least two sides. A removable top can help with handling and training. Transport cats in a carrier rather than loose in a car.""",
         "箱体稳固、可清洁、可舒适调整姿势，至少两侧通风；乘车不能让猫自由散放。"),
    _doc("CAT16", "Removing hazards from a cat's room", "清除房间危险物", "routine_care", "home_safety", "home_safety", "Here's a step-by-step guide on setting up a space for your new cat",
         """Cats Protection recommends a private starting space kept away from dogs, children and guests. Remove cleaning products and objects that could be knocked over. Add a hiding place and the cat's essential resources, together with enrichment such as puzzle toys, wand toys or cardboard boxes.""",
         "初始空间保持私密，清走清洁剂及易被打翻的物品，加入躲藏处、必需品与活动用品。"),
    _doc("CAT17", "Storing dry and opened canned food", "干粮与开封罐头储存", "routine_care", "food_storage", "cornell_feeding", "Types of Commercial Cat Food: Dry Food; Canned Food",
         """Cornell recommends storing dry cat food in a cool, dry place, checking its expiry date and using an airtight container to help preserve nutrients and flavour. Prolonged storage can degrade vitamins and allow fats to become rancid. Refrigerate unused portions of opened canned food to preserve quality and reduce spoilage.""",
         "干粮阴凉干燥密封保存并检查到期日；开封罐头的未用部分应冷藏。"),
    _doc("CAT18", "Loss of appetite needs veterinary assessment", "不进食需专业评估", "health_concern", "loss_of_appetite", "anorexia", "Main article: causes and veterinary assessment",
         """Cornell explains that appetite loss is a sign with many possible physical or psychological causes, rather than a diagnosis. A cat that is not eating deserves veterinary assessment. Persistent food refusal can cause serious harm, especially in very young kittens. An owner should not assume the cat is merely choosing to be difficult.""",
         "食欲丧失是症状而非诊断；不进食应接受兽医评估，幼猫尤其脆弱。"),
    _doc("CAT19", "Breathing difficulty requires immediate help", "呼吸困难需立即求助", "health_concern", "breathing_difficulty", "dyspnea", "Main article: signs of dyspnea and advice to owners",
         """Cornell identifies difficult breathing, unusually rapid breathing and noisy open-mouth breathing as possible signs of respiratory distress. Breathing difficulty is a clinical sign, not a specific diagnosis. If a cat cannot breathe comfortably, seek veterinary help immediately rather than trying to resolve it at home; determining the cause requires professional examination.""",
         "呼吸困难及张口喘气等可提示呼吸窘迫，应立即求助；不能据此自行诊断或在家处理。"),
    _doc("CAT20", "Straining with little or no urine", "用力排尿却尿量极少", "health_concern", "suspected_urinary_obstruction", "urinary", "Urethral Obstruction",
         """Cornell describes repeated attempts to urinate, straining and little or no urine as signs that can occur with urethral obstruction. Other urinary problems can look similar, so symptoms alone cannot establish a diagnosis. Suspected obstruction requires immediate veterinary attention because it is potentially life-threatening. Male cats have greater obstruction risk.""",
         "反复用力却几乎无尿可能涉及尿道阻塞，疑似阻塞应立即求助；相似症状不能自行确诊。"),
]

# 与原 Notebook 的 DOCS 格式兼容：检索仅使用英文标题和英文正文。
DOCS = [f"{item['title']}. {item['text']}" for item in KNOWLEDGE_BASE]

# retrieve() 返回的整数 doc_id 用来查这里，不要拿本次 Top-3 的排名当文档编号。
DOC_META = {
    doc_id: {key: value for key, value in item.items() if key != "text"}
    for doc_id, item in enumerate(KNOWLEDGE_BASE)
}

# 以下只是演示/开发问题，并不是未经使用的最终留出测试集。
# 检查 Part 2 能不能“找到正确资料，再根据资料回答”的五道测试题
EXAMPLE_QUESTIONS = [
    {
        "question": "According to the selected Cats Protection notes, should new and old cat food be mixed, and how long should the transition take?",
        "expected_stable_ids": ["CAT02"],
    },
    {
        "question": "Which water bowl materials and positions are suggested in the selected Cats Protection notes?",
        "expected_stable_ids": ["CAT03"],
    },
    {
        "question": "According to the selected litter-tray notes, how often should deposits be removed and the whole tray washed?",
        "expected_stable_ids": ["CAT06"],
    },
    {
        "question": "What sequence of barriers does the selected introduction guide recommend before cats meet face to face?",
        "expected_stable_ids": ["CAT13"],
    },
    {
        "question": "According to the selected carrier guide, how many sides should have ventilation openings?",
        "expected_stable_ids": ["CAT15"],
    },
]

if __name__ == "__main__":
    print(f"{len(DOCS)} documents from {len(SOURCE_PAGES)} source pages")

20 documents from 15 source pages


### Chunking
Long documents get cut into overlapping slices before embedding. Why: one embedding holds one
*average* meaning, so a long multi-topic document becomes a blurry vector that matches everything
a little and nothing well. Small chunks keep each vector about a single idea. The **overlap** stops
an idea that straddles a boundary from being cut in half.

This is a knob you tune against your evidence, not a setting with a correct value.

In [6]:
# TRY THIS (knob): watch what changes in Section 3 when you move these.
CHUNK_WORDS = 70     # try 15 (tiny) or 300 (whole document = one chunk)
OVERLAP     = 12     # words shared with the next chunk

def chunk(text, size=CHUNK_WORDS, overlap=OVERLAP):
    words, out, i = text.split(), [], 0
    while i < len(words):
        out.append(' '.join(words[i:i+size]))
        if i + size >= len(words):
            break
        i += max(1, size - overlap)
    return out

CHUNKS = [(c, d_i) for d_i, d in enumerate(DOCS) for c in chunk(d)]
print(f'{len(DOCS)} docs -> {len(CHUNKS)} chunks (CHUNK_WORDS={CHUNK_WORDS}, OVERLAP={OVERLAP})')

20 docs -> 20 chunks (CHUNK_WORDS=70, OVERLAP=12)


### Embed and retrieve
The embedder runs locally — a small model downloaded once, no key, no cost. If the download fails
(no internet on the machine, or a locked-down environment), the fallback cell below uses **TF-IDF**
word-overlap instead. TF-IDF works, and it is a useful contrast: it matches *words*, not *meaning*,
so it fails on paraphrase. If you use the fallback, say so in your writeup — it changes which
failures you find.

In [7]:
EMBEDDER = 'sentence-transformers/all-MiniLM-L6-v2'
USING = None
try:
    from sentence_transformers import SentenceTransformer
    _m = SentenceTransformer(EMBEDDER)
    def embed(texts):
        v = _m.encode(list(texts), normalize_embeddings=True)
        return np.asarray(v)
    USING = f'local embeddings ({EMBEDDER}) — matches MEANING'
except Exception as e:
    print('embedder unavailable ->', str(e)[:120])
    from sklearn.feature_extraction.text import TfidfVectorizer
    _v = TfidfVectorizer().fit([c for c, _ in CHUNKS])
    def embed(texts):
        m = _v.transform(list(texts)).toarray()
        n = np.linalg.norm(m, axis=1, keepdims=True)
        return m / np.where(n == 0, 1, n)
    USING = 'TF-IDF fallback — matches WORDS, not meaning'

print('using:', USING)
MATRIX = embed([c for c, _ in CHUNKS])

TOP_K = 3            # TRY THIS (knob): 1 (brittle) or 8 (noisy)

def retrieve(question, k=TOP_K):
    q = embed([question])[0]
    scores = MATRIX @ q
    order = np.argsort(-scores)[:k]
    return [(CHUNKS[i][0], CHUNKS[i][1], float(scores[i])) for i in order]

for txt, d_i, s in retrieve(
    "According to Cats Protection, how often should litter "
    "tray deposits be removed and the whole tray washed?"
):
    print(f'  {s:.3f}  [doc {d_i}]  {txt[:120]}...')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

using: local embeddings (sentence-transformers/all-MiniLM-L6-v2) — matches MEANING
  0.819  [doc 5]  Cleaning litter trays. Cats Protection advises removing urine and faeces twice daily, or more often if needed. Empty and...
  0.579  [doc 4]  Litter tray quantity and placement. Cats Protection recommends one litter tray for each cat plus one additional tray. Pu...
  0.452  [doc 2]  Water bowls and drinking preferences. Cats Protection suggests wide ceramic, metal or glass water bowls so whiskers are ...


### Answer from the retrieved text — and only from it
The instruction to say *"the notes do not say"* is the whole point. Without it the model fills gaps
from its training data and you cannot tell which sentences came from your documents. That is
**silent failure**: a confident, fluent, unsourced answer.

In [8]:
GROUNDED = (
    "Answer using ONLY the notes provided. "
    "If the notes do not contain the answer, reply exactly: The notes do not say. "
    "Do not use any other knowledge. Quote the phrase you relied on."
)

def answer(question, k=TOP_K, show=True):
    hits = retrieve(question, k)
    context = '\n\n'.join(f'[{i+1}] {t}' for i, (t, _, _) in enumerate(hits))
    out = generate(f'NOTES:\n{context}\n\nQUESTION: {question}', system=GROUNDED)
    if show:
        print(f'Q: {question}\nA: {out}\n   (top chunk score {hits[0][2]:.3f}, doc {hits[0][1]})\n')
    return out, hits

answer(
    "According to Cats Protection, how often should litter "
    "tray deposits be removed and the whole tray washed?"
)

Q: According to Cats Protection, how often should litter tray deposits be removed and the whole tray washed?
A: "removing urine and faeces twice daily, or more often if needed. Empty and wash the tray with hot soapy water or a cat-safe cleaner at least weekly."
   (top chunk score 0.819, doc 5)



('"removing urine and faeces twice daily, or more often if needed. Empty and wash the tray with hot soapy water or a cat-safe cleaner at least weekly."',
 [('Cleaning litter trays. Cats Protection advises removing urine and faeces twice daily, or more often if needed. Empty and wash the tray with hot soapy water or a cat-safe cleaner at least weekly. Covered trays may need thorough cleaning more often because they retain smells. Avoid newspaper or plastic liners inside the tray.',
   5,
   0.8187583684921265),
  ('Litter tray quantity and placement. Cats Protection recommends one litter tray for each cat plus one additional tray. Put trays in separate quiet, accessible locations, away from food, water and sleeping areas. Avoid noisy appliances and busy passageways. A row of neighbouring trays is not equivalent to distributing them. Cats with mobility difficulties benefit from low-sided trays.',
   4,
   0.5787413120269775),
  ('Water bowls and drinking preferences. Cats Protection sugg

---
## 2 · Five questions it could not answer without your documents  ### REPLACE ME

Each question should have a **specific** answer that lives in your documents — a date, a name, a
number, a decision and its outcome. "What is a monstera?" is a bad question: the model knows that
already, so a correct answer proves nothing about your retrieval.

In [9]:
# 从新知识库代码中读取五个演示问题
QUESTIONS = [
    item["question"]
    for item in EXAMPLE_QUESTIONS
]

# 依次检索资料并回答
for q in QUESTIONS:
    out, hits = answer(q)

    # 显示检索到的资料编号，方便核对
    print(
        "Retrieved documents:",
        [DOC_META[doc_id]["stable_id"] for _, doc_id, _ in hits]
    )
    print()

Q: According to the selected Cats Protection notes, should new and old cat food be mixed, and how long should the transition take?
A: The notes say: "Cats Protection advises offering a small amount of new food beside the usual food in separate bowls, rather than mixing them. If the cat accepts it, increase the new food and reduce the old food over a week or longer."
   (top chunk score 0.727, doc 1)

Retrieved documents: ['CAT02', 'CAT01', 'CAT17']

Q: Which water bowl materials and positions are suggested in the selected Cats Protection notes?
A: "wide ceramic, metal or glass water bowls so whiskers are not pressed against the sides. Keep water separate from food and litter trays, and offer bowls in several locations."
   (top chunk score 0.717, doc 2)

Retrieved documents: ['CAT03', 'CAT14', 'CAT10']

Q: According to the selected litter-tray notes, how often should deposits be removed and the whole tray washed?
A: "removing urine and faeces twice daily, or more often if needed. Empty

---
## 3 · Break it  ### REPLACE ME

**This section carries the marks.** You are looking for a question where retrieval brings back the
wrong passage — not where the model writes a clumsy sentence.

Four kinds of break that reliably work. Try all four; keep the one that fails most interestingly.

| Kind | What to try |
|---|---|
| **Vocabulary mismatch** | Ask using a word that never appears in your documents but means the same thing. Devastating for TF-IDF; embeddings usually survive it. |
| **Two documents needed** | Ask something that requires combining facts from two documents. Top-k retrieval fetches the *most similar* chunks, which is not the same as *the set that answers the question*. |
| **Distractor** | Ask about a topic covered in several documents where only one is relevant. Watch the right chunk get outvoted. |
| **Absent but plausible** | Ask something your documents nearly, but do not quite, cover. Does it say "the notes do not say", or does it invent? |

**Diagnose, do not just report.** "It got it wrong" is a Developing answer. "The right chunk ranked
fourth with score 0.31 while an irrelevant chunk scored 0.44 because both contain the word
*water*" is a Proficient one.

In [11]:
BREAKERS = [
    (
        "vocabulary mismatch",
        "According to the notes, how often should I scoop "
        "wee and poo out of my kitty's loo, and how often "
        "should I give the entire container a proper wash?"
    ),

    (
        "two documents needed",
        "According to the notes, how should I introduce "
        "a replacement cat food, and how should I store "
        "the unused portion of an opened can?"
    ),

    (
        "distractor",
        "I am arranging food bowls, water bowls, beds and "
        "litter trays for several cats. Specifically for "
        "drinking water, what bowl materials and bowl "
        "width do the notes recommend?"
    ),

    (
        "absent but plausible",
        "According to the notes, what is the exact price "
        "in Singapore dollars of the cat carrier "
        "recommended by Cats Protection?"
    ),
]

for kind, q in BREAKERS:
    print(f"\n=== {kind.upper()} ===")

    out, hits = answer(q)

    print("Retrieved passages:")
    for text, doc_id, score in hits:
        meta = DOC_META[doc_id]
        print(
            f"\n{score:.3f} | "
            f"{meta['stable_id']} | {meta['title']}"
        )
        print(text)

    print()


=== VOCABULARY MISMATCH ===
Q: According to the notes, how often should I scoop wee and poo out of my kitty's loo, and how often should I give the entire container a proper wash?
A: According to the notes, you should "remove urine and faeces twice daily, or more often if needed" and "empty and wash the tray with hot soapy water or a cat-safe cleaner at least weekly."
   (top chunk score 0.562, doc 5)

Retrieved passages:

0.562 | CAT06 | Cleaning litter trays
Cleaning litter trays. Cats Protection advises removing urine and faeces twice daily, or more often if needed. Empty and wash the tray with hot soapy water or a cat-safe cleaner at least weekly. Covered trays may need thorough cleaning more often because they retain smells. Avoid newspaper or plastic liners inside the tray.

0.352 | CAT03 | Water bowls and drinking preferences
Water bowls and drinking preferences. Cats Protection suggests wide ceramic, metal or glass water bowls so whiskers are not pressed against the sides. Keep

In [12]:
# 同一个问题：换粮方法 + 开封罐头储存
q = BREAKERS[1][1]

for k in [3, 1]:
    print(f"\n=== SAME QUESTION: TOP-K={k} ===")

    out, hits = answer(q, k=k)

    print("\nRetrieved passages:")
    for text, doc_id, score in hits:
        meta = DOC_META[doc_id]
        print(
            f"\n{score:.3f} | "
            f"{meta['stable_id']} | {meta['title']}"
        )
        print(text)

    print()


=== SAME QUESTION: TOP-K=3 ===
Q: According to the notes, how should I introduce a replacement cat food, and how should I store the unused portion of an opened can?
A: To introduce a replacement cat food, "offer a small amount of new food beside the usual food in separate bowls, rather than mixing them. If the cat accepts it, increase the new food and reduce the old food over a week or longer." For storing the unused portion of an opened can, "refrigerate unused portions of opened canned food to preserve quality and reduce spoilage."
   (top chunk score 0.683, doc 16)


Retrieved passages:

0.683 | CAT17 | Storing dry and opened canned food
Storing dry and opened canned food. Cornell recommends storing dry cat food in a cool, dry place, checking its expiry date and using an airtight container to help preserve nutrients and flavour. Prolonged storage can degrade vitamins and allow fats to become rancid. Refrigerate unused portions of opened canned food to preserve quality and reduce sp

### Diagnose the one you will submit
Fill this in **in prose**, from the scores printed above. This is your documented failure.

In [14]:
FAILURE_DIAGNOSIS = """
Question:
According to the notes, how should I introduce a replacement
cat food, and how should I store the unused portion of an
opened can?

Configuration that failed:
Top-K=1. The default Top-K=3 configuration answered this
question successfully.

Observed answer:
The notes do not say.

Expected answer:
Offer the new and old food in separate bowls and gradually
change the amounts over a week or longer. Refrigerate the
unused portion of opened canned food.

Evidence and ranking:
CAT17, covering food storage, ranked first with score 0.683.
CAT02, covering food transition, ranked second with score 0.555.
CAT01, covering food selection, ranked third with score 0.465.

Failure mechanism:
The question required evidence from two documents.
Top-K=1 retrieved CAT17 but excluded CAT02, so the model
could not support the food-transition part of the answer.
The grounding prompt required a fixed refusal when the
notes did not contain the answer. Consequently, the model
declined the entire question, including the storage part
that the retrieved passage could support.

Comparison:
With Top-K=3, both required documents were retrieved and
the model answered both parts using their content.

Trade-off:
Keeping Top-K=3 resolved this case but included an additional
passage, CAT01, that was unnecessary for the answer.
Retrieving more passages adds input tokens and may introduce
distraction. This single comparison does not establish that
Top-K=3 is optimal for every question.
"""

print(FAILURE_DIAGNOSIS)


Question:
According to the notes, how should I introduce a replacement
cat food, and how should I store the unused portion of an
opened can?

Configuration that failed:
Top-K=1. The default Top-K=3 configuration answered this
question successfully.

Observed answer:
The notes do not say.

Expected answer:
Offer the new and old food in separate bowls and gradually
change the amounts over a week or longer. Refrigerate the
unused portion of opened canned food.

Evidence and ranking:
CAT17, covering food storage, ranked first with score 0.683.
CAT02, covering food transition, ranked second with score 0.555.
CAT01, covering food selection, ranked third with score 0.465.

Failure mechanism:
The question required evidence from two documents.
Top-K=1 retrieved CAT17 but excluded CAT02, so the model
could not support the food-transition part of the answer.
The grounding prompt required a fixed refusal when the
notes did not contain the answer. Consequently, the model
declined the entire questi

---
## 4 · Two checks worth running before you write

**Check 1 — is retrieval doing anything?** Ask the same five questions with the documents removed.
If the model still answers correctly, your documents are too generic and Part 2 has not been
demonstrated. Fix the documents.

**Check 2 — does the grounding instruction hold?** A system that invents when it should decline is
worse than one that declines too often, because you cannot see it happening.

In [10]:
print('=== CHECK 1: same questions, NO documents ===')
for q in QUESTIONS:
    out = generate(
        q,
        system="Answer in one short sentence.",
        max_new_tokens=80
    )
    print(f"Q: {q}\nA: {out}\n")

print('=== CHECK 2: something your documents definitely do not cover ===')
answer('What is the name of the officer who approved the PCSS programme?')

=== CHECK 1: same questions, NO documents ===
Q: According to the selected Cats Protection notes, should new and old cat food be mixed, and how long should the transition take?
A: New and old cat food should be mixed gradually over a period of about 7 to 10 days for a smooth transition.

Q: Which water bowl materials and positions are suggested in the selected Cats Protection notes?
A: The Cats Protection notes suggest using ceramic or stainless steel bowls and placing them in quiet, accessible locations away from food and litter boxes.

Q: According to the selected litter-tray notes, how often should deposits be removed and the whole tray washed?
A: Deposits should be removed daily, and the whole tray should be washed weekly.

Q: What sequence of barriers does the selected introduction guide recommend before cats meet face to face?
A: The recommended sequence of barriers includes scent swapping, visual barriers, and finally, supervised meetings.

Q: According to the selected carrier g

('The notes do not say.',
 [('Introducing cats in stages. Cats Protection describes staged introductions: a separate sanctuary room, scent exchange using cloths, visual contact through a glass barrier, then mesh-barrier contact before short face-to-face meetings. Progress only when cats are comfortable with the preceding stage. Do not force approach or meetings, and provide ways to retreat or reach a high place.',
   12,
   0.03808946907520294),
  ("Complete and complementary cat food. Cats Protection recommends food labelled complete because it supplies the nutrients a cat needs. Complementary products, including treats, do not provide a balanced diet by themselves. Food should suit the cat's life stage. Homemade meals are not recommended unless a veterinarian advises them for a medical reason.",
   0,
   0.03195013850927353),
  ('Cleaning litter trays. Cats Protection advises removing urine and faeces twice daily, or more often if needed. Empty and wash the tray with hot soapy water 

---
## 5 · YOUR TURN

1. Replace `DOCS` with 10–30 documents from your domain that contain things a model cannot know.
2. Replace `QUESTIONS` with five that are answerable only from them. Run and read every answer.
3. Run Check 1. If the model answers without your documents, go back to step 1.
4. Work through all four `BREAKERS`. Keep the most interesting failure and diagnose it properly.
5. Try `CHUNK_WORDS`, `OVERLAP` and `TOP_K` at extremes. Note which failures appear and disappear.

### What your ~400 words must contain

- What your documents are, and why a model could not answer these questions without them.
- The five worked examples (the notebook output is enough; the words are for what they show).
- **The failure, diagnosed** — which chunk should have won, which did, the scores, and the
  mechanism.
- What you would change to fix it, and **what that fix would break** — bigger chunks improve
  context and dilute matching; a larger top-k improves recall and adds noise and tokens. There is
  no free setting, and saying so is the postgraduate move.

In [ ]:
print(f'tokens this session: {TOKENS["in"]} in / {TOKENS["out"]} out')

tokens this session: 3347 in / 604 out
